# Camada Gold - Visões de negócio da UTI Neonatal

Este notebook transforma os dados tratados e armaqzenados na camada Silver em visoes para Mariana (enfermeira), Dr. Rafael (neonatologista) e Alice (gestora hospitalar).

**Importante:** os alertas abaixo sao regras operacionais demonstrativas e nao substituem avaliacao medica. 

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

pasta_projeto = Path.cwd()
if pasta_projeto.name in {"gold", "silver"}:
    pasta_projeto = pasta_projeto.parent
pasta_silver = pasta_projeto / "silver"
pasta_gold = pasta_projeto / "gold"
pasta_gold.mkdir(parents=True, exist_ok=True)

leituras = pd.read_csv(pasta_silver / "leituras_sensores.csv", parse_dates=["data_hora"])
leituras = leituras.sort_values("data_hora").reset_index(drop=True)
leituras["data"] = leituras["data_hora"].dt.date.astype(str)
leituras["hora"] = leituras["data_hora"].dt.hour
leituras["periodo"] = leituras["data_hora"].dt.floor("15min")
pasta_graficos = pasta_gold / "graficos"
pasta_graficos.mkdir(parents=True, exist_ok=True)
leituras.head() 

## 1. Visao da enfermeira: monitoramento e priorizacao

A tabela abaixo resume a leitura mais recente de cada sensor e permite uma triagem inicial. Como não existe identificacao de paciente ou incubadora, a coluna `alvo_monitoramento` identifica o tipo de sensor.

In [ ]:
# Faixas operacionais de demonstracao. Validar e substituir pelos limites institucionais.
FAIXAS_OPERACIONAIS = {
    "temperatura": (20.0, 30.0),
    "oxigenio": (92.0, 100.0),
    "frequencia_cardiaca": (60.0, 100.0),
    "umidade": (40.0, 80.0),
}

def classificar_alerta(linha):
    if linha["sensor"] == "ruido":
        if linha["valor"] > 20:
            return "alta"
        if linha["valor"] > 15:
            return "media"
        return "normal"
    minimo, maximo = FAIXAS_OPERACIONAIS.get(linha["sensor"], (None, None))
    if minimo is None:
        return "nao_classificado"
    if linha["valor"] < minimo or linha["valor"] > maximo:
        return "alta"
    return "normal"

ultima_leitura = leituras.groupby("sensor", as_index=False).tail(1).copy()
ultima_leitura["prioridade"] = ultima_leitura.apply(classificar_alerta, axis=1)
ultima_leitura["alvo_monitoramento"] = ultima_leitura["sensor"].map(
    lambda sensor: "incubadora" if sensor in {"temperatura", "umidade", "ruido"} else "dispositivo neonatal"
)
visao_enfermeira = ultima_leitura[[
    "data_hora", "alvo_monitoramento", "sensor", "valor", "unidade", "prioridade"
]].sort_values(["prioridade", "sensor"])
visao_enfermeira.to_csv(pasta_gold / "visao_enfermeira_ultima_leitura.csv", index=False)
visao_enfermeira

In [ ]:
alertas = leituras.copy()
alertas["prioridade"] = alertas.apply(classificar_alerta, axis=1)
alertas = alertas[alertas["prioridade"] != "normal"].copy()
alertas["alvo_monitoramento"] = alertas["sensor"].map(
    lambda sensor: "incubadora" if sensor in {"temperatura", "umidade", "ruido"} else "dispositivo neonatal"
)
visao_alertas = alertas[[
    "data_hora", "alvo_monitoramento", "sensor", "valor", "unidade", "prioridade"
]].sort_values("data_hora", ascending=False)
visao_alertas.to_csv(pasta_gold / "alertas_operacionais.csv", index=False)
visao_alertas.head(20)

## 2. Visao do neonatologista: historico e tendencias

A agregacao por hora reduz o volume de dados sem perder a tendência. A correlacao abaixo é temporal e exploratória: ela não prova relacao clínica nem associa um sensor a um paciente específico.

In [ ]:
historico_horario = leituras.set_index("data_hora").groupby("sensor")["valor"].resample("1h").agg(
    quantidade="count", media="mean", minimo="min", maximo="max", desvio_padrao="std"
).reset_index()
historico_horario.to_csv(pasta_gold / "historico_horario_sensores.csv", index=False)
historico_horario.head(10)

In [ ]:
serie_temporal = leituras.pivot_table(index="periodo", columns="sensor", values="valor", aggfunc="mean")
correlacoes_temporais = serie_temporal.corr().reset_index().melt(
    id_vars="sensor", var_name="sensor_relacionado", value_name="correlacao"
)
correlacoes_temporais = correlacoes_temporais[correlacoes_temporais["sensor"] != correlacoes_temporais["sensor_relacionado"]]
correlacoes_temporais.to_csv(pasta_gold / "correlacoes_temporais_sensores.csv", index=False)
correlacoes_temporais.sort_values("correlacao", ascending=False).head(10)

## 3. Visao da gestora: indicadores operacionais

Os indicadores mostram volume, cobertura temporal, completude e quantidade de alertas. O tempo médio de resposta não é calculado porque não existem registros de abertura e encerramento de atendimentos.

In [ ]:
indicadores_sensor = leituras.groupby(["data", "sensor", "unidade"], as_index=False).agg(
    leituras=("valor", "size"),
    media=("valor", "mean"),
    minimo=("valor", "min"),
    maximo=("valor", "max"),
    primeiro_registro=("data_hora", "min"),
    ultimo_registro=("data_hora", "max"),
)
alertas_por_sensor = alertas.groupby(["data", "sensor"], as_index=False).size().rename(columns={"size": "alertas"})
indicadores_gestao = indicadores_sensor.merge(alertas_por_sensor, on=["data", "sensor"], how="left")
indicadores_gestao["alertas"] = indicadores_gestao["alertas"].fillna(0).astype(int)
indicadores_gestao["tempo_resposta_medio_minutos"] = pd.NA
indicadores_gestao.to_csv(pasta_gold / "indicadores_gestao.csv", index=False)
indicadores_gestao

In [ ]:
resumo_execucao = pd.DataFrame({
    "indicador": [
        "leituras_processadas", "sensores_monitorados", "alertas_operacionais",
        "inicio_coleta", "fim_coleta", "tempo_resposta_medio_minutos"
    ],
    "valor": [
        len(leituras), leituras["sensor"].nunique(), len(alertas),
        leituras["data_hora"].min(), leituras["data_hora"].max(),
        "indisponivel: faltam eventos de atendimento"
    ]
})
resumo_execucao.to_csv(pasta_gold / "resumo_execucao_gold.csv", index=False)
resumo_execucao

## 4. Gráficos

Os gráficos abaixo apoiam a leitura rápida da equipe assistencial, a análise de tendências clínicas e o acompanhamento operacional da gestão. 

In [ ]:
# Mariana: painel compacto da leitura mais recente por sensor.
cores = ultima_leitura["prioridade"].map({"alta": "#c0392b", "media": "#f39c12", "normal": "#27ae60"}).fillna("#7f8c8d")
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(ultima_leitura["sensor"], ultima_leitura["valor"], color=cores)
ax.set_title("Monitoramento atual por sensor")
ax.set_xlabel("Sensor")
ax.set_ylabel("Valor medido")
ax.tick_params(axis="x", rotation=25)
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(pasta_graficos / "mariana_ultima_leitura.png", dpi=150)
plt.show()
plt.close(fig)

In [ ]:
# Mariana: quantidade de alertas operacionais por sensor e prioridade.
contagem_alertas = alertas.groupby(["sensor", "prioridade"]).size().unstack(fill_value=0)
contagem_alertas = contagem_alertas.reindex(columns=["alta", "media"], fill_value=0)
fig, ax = plt.subplots(figsize=(10, 5))
contagem_alertas.plot(kind="bar", stacked=True, color={"alta": "#c0392b", "media": "#f39c12"}, ax=ax)
ax.set_title("Alertas operacionais por sensor")
ax.set_xlabel("Sensor")
ax.set_ylabel("Quantidade de alertas")
ax.tick_params(axis="x", rotation=25)
ax.legend(title="Prioridade")
fig.tight_layout()
fig.savefig(pasta_graficos / "mariana_alertas_por_sensor.png", dpi=150)
plt.show()
plt.close(fig)

In [ ]:
# Dr. Rafael: evoluçãoo horária, com um painel por parâmetro.
sensores = historico_horario["sensor"].unique()
fig, eixos = plt.subplots(len(sensores), 1, figsize=(12, 3 * len(sensores)), sharex=True)
for eixo, sensor in zip(eixos, sensores):
    serie = historico_horario[historico_horario["sensor"] == sensor]
    eixo.plot(serie["data_hora"], serie["media"], marker="o", markersize=3, label=sensor)
    eixo.fill_between(serie["data_hora"], serie["minimo"], serie["maximo"], alpha=0.15)
    eixo.set_ylabel(sensor)
    eixo.grid(alpha=0.25)
    eixo.legend(loc="upper left")
eixos[-1].set_xlabel("Data e hora")
fig.suptitle("Tendencia horaria dos parametros monitorados", y=1.0)
fig.tight_layout()
fig.savefig(pasta_graficos / "rafael_tendencias_horarias.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
# Dr. Rafael: mapa das correlações temporais exploratórias.
matriz_correlacao = serie_temporal.corr()
fig, ax = plt.subplots(figsize=(8, 6))
imagem = ax.imshow(matriz_correlacao, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(matriz_correlacao.columns)), matriz_correlacao.columns, rotation=35, ha="right")
ax.set_yticks(range(len(matriz_correlacao.index)), matriz_correlacao.index)
for linha in range(len(matriz_correlacao.index)):
    for coluna in range(len(matriz_correlacao.columns)):
        ax.text(coluna, linha, f"{matriz_correlacao.iloc[linha, coluna]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(imagem, ax=ax, label="Correlacao")
ax.set_title("Correlacao temporal entre sensores")
fig.tight_layout()
fig.savefig(pasta_graficos / "rafael_correlacao_temporal.png", dpi=150)
plt.show()
plt.close(fig)

In [ ]:
# Alice: indicadores de volume e alertas por sensor.
resumo_sensores = indicadores_gestao.groupby("sensor", as_index=False).agg(
    leituras=("leituras", "sum"), alertas=("alertas", "sum")
)
fig, eixo_leituras = plt.subplots(figsize=(10, 5))
eixo_alertas = eixo_leituras.twinx()
posicoes = range(len(resumo_sensores))
eixo_leituras.bar([p - 0.2 for p in posicoes], resumo_sensores["leituras"], width=0.4, label="Leituras", color="#3498db")
eixo_alertas.bar([p + 0.2 for p in posicoes], resumo_sensores["alertas"], width=0.4, label="Alertas", color="#e67e22")
eixo_leituras.set_xticks(list(posicoes), resumo_sensores["sensor"], rotation=25)
eixo_leituras.set_ylabel("Leituras")
eixo_alertas.set_ylabel("Alertas")
eixo_leituras.set_title("Indicadores operacionais por sensor")
eixo_leituras.grid(axis="y", alpha=0.25)
eixo_leituras.legend(loc="upper left")
eixo_alertas.legend(loc="upper right")
fig.tight_layout()
fig.savefig(pasta_graficos / "alice_indicadores_operacionais.png", dpi=150)
plt.show()
plt.close(fig)